In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    precision_recall_curve, roc_curve, average_precision_score
)
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Load data
df=pd.read_csv('/Users/mohammadhosein/Desktop/churn_project/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Phase 1: Business Understanding (already covered in previous response)

# Phase 2: Data Understanding & Phase 3: Data Preparation
# Handle missing values (including spaces in TotalCharges)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

# Separate features and target
X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Define feature types
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
binary_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
categorical_features = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                        'Contract', 'PaymentMethod']

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('scaler', RobustScaler())
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('label', LabelEncoder())
])

# For multi-category features, we'll one-hot encode in ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', 'passthrough', categorical_features),  # Will be handled separately
        ('binary', 'passthrough', binary_features)
    ])

# Phase 4: Modeling
# Define models with their respective preprocessing steps
# First, let's prepare the data properly

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X[categorical_features], drop_first=False)
X_binary = X[binary_features].apply(lambda col: col.map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}))
X_numeric_scaled = preprocessor.fit_transform(X[numeric_features])

# Combine all features
X_processed = np.hstack([X_numeric_scaled, X.binary.values, X_encoded.values])

# Train-test split (stratified to handle class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Model Definitions
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# Phase 5: Hyperparameter Tuning
param_grids = {
    'Logistic Regression': {
        'C': [0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    },
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
        'learning_rate': [0.1, 0.01],
        'scale_pos_weight': [1, 3]  # Handle class imbalance
    }
}

# Fit models with hyperparameter tuning
best_models = {}
for name, model in models.items():
    print(f"\nTuning {name}...")
    search = GridSearchCV(model, param_grids[name], cv=3, scoring='recall', n_jobs=-1)
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    print(f"Best params for {name}: {search.best_params_}")

# Phase 6: Evaluation
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    
    # Metrics
    report = classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'])
    auc_roc = roc_auc_score(y_test, y_pred_proba)
    avg_precision = average_precision_score(y_test, y_pred_proba)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    
    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    
    return {
        'model_name': model_name,
        'classification_report': report,
        'auc_roc': auc_roc,
        'avg_precision': avg_precision,
        'confusion_matrix': cm,
        'roc_curve': (fpr, tpr),
        'pr_curve': (precision, recall)
    }

# Evaluate all models
results = {}
for name, model in best_models.items():
    results[name] = evaluate_model(model, X_test, y_test, name)

# Visualization of Results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

# Plot ROC curves
for name, result in results.items():
    axes[0].plot(result['roc_curve'][0], result['roc_curve'][1], label=f"{name} (AUC = {result['auc_roc']:.2f})")
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

# Plot Precision-Recall curves
for name, result in results.items():
    axes[1].plot(result['pr_curve'][1], result['pr_curve'][0], label=f"{name} (AP = {result['avg_precision']:.2f})")
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

# Plot Confusion Matrices
for idx, (name, result) in enumerate(results.items()):
    if idx < 2:  # Only plot first two
        sns.heatmap(result['confusion_matrix'], annot=True, fmt='d', ax=axes[2+idx], cmap='Blues')
        axes[2+idx].set_title(f'{name} Confusion Matrix')
        axes[2+idx].set_xlabel('Predicted')
        axes[2+idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Print detailed reports
for name, result in results.items():
    print(f"\n{name} Performance:")
    print("="*30)
    print(result['classification_report'])
    print(f"AUC-ROC: {result['auc_roc']:.3f}")
    print(f"Average Precision: {result['avg_precision']:.3f}")